In [ ]:
# =========================
# Cell 1: Install packages
# =========================
!pip install -q kaggle scikit-learn matplotlib seaborn

In [ ]:
# =========================
# Cell 2: Kaggle API setup
# =========================
from google.colab import files
_ = files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# =========================
# Cell 3: Download dataset
# =========================
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip -q brain-tumor-mri-dataset.zip -d /content/brain_tumor_dataset
!ls -R /content/brain_tumor_dataset | head -n 50

In [ ]:
# =====================================
# Cell 4: Create train/val/test loaders
# =====================================
import os, json, math
import tensorflow as tf

base_dir = '/content/brain_tumor_dataset'
train_root = os.path.join(base_dir, 'Training')  # we'll split this into train/val
test_dir = os.path.join(base_dir, 'Testing')

IMG_SIZE = (300, 300)  # EfficientNetB3 native size
BATCH = 32
SEED = 42
VAL_SPLIT = 0.15

# IMPORTANT: build train+val from the SAME root to guarantee class_names order
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_root,
    validation_split=VAL_SPLIT,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode='int',
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_root,
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode='int',
    shuffle=False
)

# Test loader (keeps same order as train_ds.class_names)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode='int',
    shuffle=False
)

CLASS_NAMES = train_ds.class_names
print("Class names (order):", CLASS_NAMES)

# Save class names for inference (avoid wrong label mapping!)
with open('/content/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)

# Caching/prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1024, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
# =====================================
# Cell 5: Class weights (handle imbalance)
# =====================================
from collections import Counter
import numpy as np

# Count labels from the (unbatched) training dataset once
train_labels_list = []
for _, y in train_ds.unbatch().take(1000000):  # big cap to cover all
    train_labels_list.append(int(y.numpy()))

counts = Counter(train_labels_list)
print("Train label counts:", counts)

num_classes = len(CLASS_NAMES)
total = sum(counts.values())
class_weight = {i: total / (num_classes * counts[i]) for i in range(num_classes)}
print("Class weights:", class_weight)

In [ ]:
# =====================================
# Cell 6: Augmentation + Model (EffNetB3)
# =====================================
from tensorflow.keras import layers, models, optimizers, losses, callbacks
from tensorflow.keras.applications.efficientnet import EfficientNetB3, preprocess_input

# Strong but safe augmentations
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
], name="augmentation")

# Backbone
base = EfficientNetB3(include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,))
base.trainable = False  # stage 1: frozen

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = layers.Lambda(preprocess_input, name='preprocess')(x)  # <- official preprocessing
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)  # numerically stable output

model = models.Model(inputs, outputs)
model.compile(
    optimizer=optimizers.Adam(1e-4),
    loss=losses.SparseCategoricalCrossentropy(),  # Removed label_smoothing
    metrics=['accuracy']
)
model.summary()

In [ ]:
# =====================================
# Cell 7: Callbacks
# =====================================
ckpt_path = "/content/best_stage1.h5"
cb = [
    callbacks.ModelCheckpoint(ckpt_path, monitor='val_loss', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1)
]

In [ ]:
# =====================================
# Cell 8: Train (stage 1: frozen)
# =====================================
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=cb,
    verbose=1
)

In [ ]:
# =====================================
# Cell 9: Fine-tune (unfreeze tail)
# =====================================
# Unfreeze last ~100 layers for fine-tuning
base.trainable = True
for layer in base.layers[:-100]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss=losses.SparseCategoricalCrossentropy(),  # Removed label_smoothing
    metrics=['accuracy']
)

ckpt_path_ft = "/content/best_stage2.h5"
cb_ft = [
    callbacks.ModelCheckpoint(ckpt_path_ft, monitor='val_loss', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4, verbose=1)
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weight,
    callbacks=cb_ft,
    verbose=1
)

In [ ]:
# =====================================
# Cell 10: Evaluate on held-out TEST set
# =====================================
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

test_probs = model.predict(test_ds)
test_pred = np.argmax(test_probs, axis=1)
test_true = np.concatenate([y for _, y in test_ds], axis=0)

print(classification_report(test_true, test_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(test_true, test_pred)
plt.figure(figsize=(7,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix - Test")
plt.show()

In [ ]:
# =====================================
# Cell 11: Save model + class names
# =====================================
model.save("/content/brain_tumor_effb3_best.h5")
with open('/content/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)
print("Saved model and class_names.json")

In [ ]:
# =====================================
# Cell 12: Single-image prediction with TTA
# =====================================
import numpy as np, json
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from google.colab import files
import matplotlib.pyplot as plt
from tensorflow.keras.applications.efficientnet import preprocess_input

# load model + class names
model = tf.keras.models.load_model(
    "/content/brain_tumor_effb3_best.h5",
    compile=False,
    custom_objects={'preprocess_input': preprocess_input}
)
with open('/content/class_names.json', 'r') as f:
    CLASS_NAMES = json.load(f)

def load_and_preprocess(path):
    img = image.load_img(path, target_size=IMG_SIZE)
    arr = image.img_to_array(img)
    return arr

# simple TTA: original + hflip + small rotations
def tta_batch(arr):
    imgs = [arr,
            tf.image.flip_left_right(arr),
            tf.image.rot90(arr, k=1),
            tf.image.rot90(arr, k=3)]
    imgs = tf.stack(imgs, axis=0)
    imgs = preprocess_input(imgs)  # IMPORTANT: same preprocessing as training
    return imgs

uploaded = files.upload()
for fn in uploaded.keys():
    arr = load_and_preprocess(fn)
    tta_imgs = tta_batch(arr)
    probs = model.predict(tta_imgs, verbose=0)
    mean_prob = np.mean(probs, axis=0)
    pred_idx = int(np.argmax(mean_prob))
    conf = float(np.max(mean_prob))

    plt.imshow(image.array_to_img(arr))
    plt.axis('off')
    title = f"Prediction: {CLASS_NAMES[pred_idx]}  |  confidence: {conf:.2f}"
    if conf < 0.60:
        title += "  (uncertain)"
    plt.title(title)
    plt.show()